In [ ]:
# --- vep path bootstrap (auto-added; safe to re-run) ---
import os, sys
from pathlib import Path
def _vep_root():
    cands = []
    _p = globals().get('__vsc_ipynb_file__')           # VSCode exposes the notebook path
    if _p: cands.append(Path(_p).resolve().parent)
    cands.append(Path.cwd().resolve())
    for s in cands:
        for c in [s, *s.parents]:                       # search upward for the repo root
            if (c / 'core').is_dir() and (c / 'classifiers').is_dir():
                return c
    return Path.cwd().resolve()
_root = _vep_root()
os.chdir(_root)                                          # data/ and outputs/ resolve from repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))                      # makes core/classifiers/app importable
# --- end vep path bootstrap ---

import os
from pathlib import Path

# Change working directory to parent directory
import sys
# Locate the repo root (folder containing the `core` package) and make it importable
_root = Path.cwd()
if not (_root / "core").is_dir():
    _root = _root.parent
os.chdir(_root)                       # data/ and outputs/ paths resolve from the repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))    # so `core`, `classifiers`, `app`, ... import in any kernel

import numpy as np 
import matplotlib.pyplot as plt
import os
import re
import pandas as pd

from core.preprocessing import load_signal_both_channels, average_two_phases, dwt_downsampling, trim, normalize_signal, artifact_removal, extract_PulseWidth_SignalPower

# 1. Raw file

In [ ]:
file = "data/Labelled_VEP_Data/RB20_RCS_LA/BC_and_RGC/RCS_RB20_3_10ms_0.59mWmm2.csv"
stim_dur, irradiance = extract_PulseWidth_SignalPower(file)
time, ch1, ch3 = load_signal_both_channels(file)

In [ ]:
time, ch3_art_removed = artifact_removal(ch1, ch3, time, t_min=0, stim_dur=stim_dur)

# Plot raw signals from both channels
plt.figure(figsize=(12, 6))

plt.subplot(3, 1, 1)
plt.plot(time, -ch1*1e-6, linewidth=2, color='blue')
plt.axvline(x=0, linestyle='dashed', color='gray')
plt.axvline(x=500, linestyle='dashed', color='gray')
plt.xlabel("Time (ms)")
plt.ylabel("Channel 1 (mV)")
plt.title("Raw Channel 1 Signal")
plt.xlim(-15, 1000)

plt.subplot(3, 1, 2)
plt.plot(time, ch3*1e-6, linewidth=2, color='red')
plt.axvline(x=0, linestyle='dashed', color='gray')
plt.axvline(x=500, linestyle='dashed', color='gray')
plt.xlabel("Time (ms)")
plt.ylabel("Channel 3 (mV)")
plt.title("Raw Channel 3 Signal")
plt.xlim(-15, 1000)

plt.subplot(3, 1, 3)
plt.plot(time, ch3_art_removed, linewidth=2, color='green')
plt.axvline(x=0, linestyle='dashed', color='gray')
plt.axvline(x=500, linestyle='dashed', color='gray')
plt.xlabel("Time (ms)")
plt.ylabel("Channel 3 - Channel 1 (mV)")
plt.title("Artifact-Removed Signal (Ch3 - Ch1)")
plt.xlim(-15, 1000)


plt.tight_layout()
plt.show()

signal = ch3_art_removed

# 2. Average two phases

In [ ]:
ch3_avg, time_avg = average_two_phases(signal, time)
print(time_avg)
plt.figure(figsize=(8, 4))
plt.plot(time_avg, ch3_avg*1e-6, linewidth=2.5, color = 'black') # Convert nV to mV for better visualization
plt.xlim(-15, 500)
plt.xlabel("Time (ms)")
plt.ylabel("Signal (mV)")
plt.show()

In [ ]:
# print how many timepoints are in the averaged signal
print(f"Number of timepoints in averaged signal: {len(time_avg)}")

# 3. Trim Signal

In [ ]:
time_trimmed, ch3_trimmed = trim(ch3_avg, time_avg, t_min=None, t_max=320)
plt.figure(figsize=(8, 4))
plt.plot(time_trimmed, ch3_trimmed*1e-6, linewidth=2.5, color = 'black') # Convert nV to mV for better visualization
plt.xlim(-15, 500)
plt.xlabel("Time (ms)")
plt.ylabel("Signal (mV)")
plt.show()

In [ ]:
print(f"Number of timepoints in trimmed signal: {len(time_trimmed)}")

# 4. DWT downsampling

In [ ]:
ch3_dwt = dwt_downsampling(ch3_trimmed, wavelet="db4", level=4)
time_dwt = np.linspace(time_trimmed[0], time_trimmed[-1], len(ch3_dwt))

plt.figure(figsize=(8, 4))
plt.plot(time_dwt, ch3_dwt*1e-6, linewidth=2.5, color = 'black') # Convert nV to mV for better visualization
plt.xlim(-15, 500)
plt.xlabel("Time (ms)")
plt.ylabel("DWT Downsampled Signal (mV)")
plt.show()

In [ ]:
print(f"Number of timepoints in dwt downsampled signal: {len(time_dwt)}")

# 5. Normalization

In [ ]:
ch3_norm = normalize_signal(ch3_dwt)
plt.figure(figsize=(6, 4))
plt.plot(time_dwt, ch3_norm, linewidth=2.5, color = 'black') # Normalized signal
plt.xlim(-15, 500)
plt.xlabel("Time (ms)")
plt.ylabel("Normalized Signal")
# plt.yticks([])
# plt.xticks([])
plt.show()

# Before vs After Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.linewidth":   1.1,
    "axes.labelsize":   11,
    "axes.titlesize":   12,
    "xtick.direction":  "out",
    "ytick.direction":  "out",
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
})

TRIM_AT = 320   # ms – must match your trim(t_max=320) call

fig = plt.figure(figsize=(8, 5))
fig.patch.set_facecolor("white")
gs = gridspec.GridSpec(2, 1, figure=fig, hspace=0.55,
                       left=0.12, right=0.95, top=0.88, bottom=0.10)

ax_raw  = fig.add_subplot(gs[0])
ax_proc = fig.add_subplot(gs[1])

# ── Raw (averaged, before preprocessing) ──────────────────────────────
ax_raw.plot(time_avg, ch3_avg * 1e-6, lw=1.8, color="black")


ax_raw.set_xlim(-15, 500)
ax_raw.set_xlabel("Time (ms)")
ax_raw.set_ylabel("Amplitude (mV)")
ax_raw.set_title("Before preprocessing", loc="left", fontweight="bold")
ax_raw.yaxis.set_major_formatter(plt.FormatStrFormatter("%.2f"))

# ── Preprocessed (DWT downsampled & normalised) ────────────────────────
ax_proc.plot(time_dwt, ch3_norm, lw=1.8, color="black")

ax_proc.set_xlim(-15, 500)
ax_proc.set_xlabel("Time (ms)")
ax_proc.set_ylabel("Normalised amplitude")
ax_proc.set_title("After preprocessing  (trimmed · DWT ↓ · normalised)", loc="left", fontweight="bold")
ax_proc.yaxis.set_major_formatter(plt.FormatStrFormatter("%.2f"))

for ax in [ax_raw, ax_proc]:
    # Shade the region that will be trimmed away
    y_min, y_max = ax.get_ylim()
    ax.axvspan(TRIM_AT, 500, color="#F89441", alpha=0.25, zorder=0)
    ax.axvline(TRIM_AT, color="#9C179E", lw=1.2, ls="--", zorder=2)
    ax.text(TRIM_AT + 5, y_max * 0.85,
                "trimmed", fontsize=8.5, color="#9C179E", style="italic")



plt.savefig("outputs/figures/preprocessing.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()